In [11]:
import numpy as np
from PIL import Image
import os
import random
from collections import Counter

def extract_features(image_path):
    """Извлекает признаки изображения."""
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img) / 255.0 
    
    brightness = np.mean(img_array)
    green_ratio = np.mean(img_array[:, :, 1]) 
    contrast = np.std(img_array)
    
    return np.array([brightness, green_ratio, contrast])

In [12]:
class KNNImageClassifier:
    def __init__(self, k=3):
        self.k = k  
        self.X_train = None  
        self.y_train = None 

    def fit(self, X_train, y_train):
        """Сохраняет обучающие данные."""
        self.X_train = X_train
        self.y_train = y_train

    def predict(self, X_test):
        """Предсказывает класс для тестовых изображений."""
        predictions = []
        for x in X_test:
            # Вычисляем расстояния до всех точек обучающей выборки
            distances = [np.linalg.norm(x - train_x) for train_x in self.X_train]
            
            # Индексы k ближайших соседей
            k_indices = np.argsort(distances)[:self.k]
            k_nearest_labels = [self.y_train[i] for i in k_indices]
            
            # Выбираем наиболее частый класс
            most_common = Counter(k_nearest_labels).most_common(1)[0][0]
            predictions.append(most_common)
        
        return predictions

In [13]:
def load_dataset(dataset_path):
    """Загружает изображения и метки классов."""
    X, y = [], []
    for class_name in ['forest', 'desert']:
        class_dir = os.path.join(dataset_path, class_name)
        for img_file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_file)
            features = extract_features(img_path)
            X.append(features)
            y.append(class_name)
    return np.array(X), np.array(y)

In [14]:
# Путь к датасету
DATASET_PATH = "origins/task_2"

# Загрузка данных
X, y = load_dataset(DATASET_PATH)

# Объединяем и перемешиваем
data = list(zip(X, y))
random.shuffle(data)  
X_shuffled, y_shuffled = zip(*data) 

# Разделение на обучающую и тестовую выборки
split_idx = int(0.8 * len(X_shuffled))
X_train, y_train = X[:split_idx], y[:split_idx]
X_test, y_test = X[split_idx:], y[split_idx:]

# Обучение KNN (k=5)
knn = KNNImageClassifier(k=5)
knn.fit(X_train, y_train)

# Предсказание для тестовых данных
y_pred = knn.predict(X_test)

# Оценка точности
accuracy = np.mean(y_pred == y_test)
print(f"Точность: {accuracy * 100:.2f}%")

Точность: 85.09%


In [17]:
img_path = 'origins/task_2/test/test_forest_2.jpg'
features = extract_features(img_path)

y_pred = knn.predict(np.array([features]))

print(y_pred[0])

forest
